# MobiMart — Mobile Retail Demand Forecasting & Inventory Optimization

**Author:** Rati Kumari Akela

## Objective

Build a data-driven system to forecast mobile phone demand and support
inventory allocation across multiple retail stores.

## Workflow

1. Generate product, store, and sales data
2. Validate and explore the dataset
3. Analyze demand patterns
4. Build a baseline forecasting model
5. Train and evaluate a Random Forest forecasting model
6. Estimate inventory requirements
7. Identify excess and shortage
8. Generate store-to-store transfer recommendations
9. Identify EOL products and markdown opportunities
10. Generate final KPIs

# 1. Import Libraries
## 2. Generate Product Data

In [3]:
import os
import pandas as pd
import random
from datetime import datetime, timedelta

# Product configuration

product_plan = {
    "Budget": {
        "count": 20,
        "min_price": 6000,
        "max_price": 15000
    },
    "Mid-range": {
        "count": 20,
        "min_price": 15001,
        "max_price": 30000
    },
    "Premium": {
        "count": 12,
        "min_price": 30001,
        "max_price": 70000
    },
    "Flagship": {
        "count": 8,
        "min_price": 70001,
        "max_price": 150000
    }
}

# Generate products

products = []

product_number = 1

for segment, details in product_plan.items():

    for i in range(details["count"]):

        model_id = f"P{product_number:03d}"

        model_name = f"Mobi {segment[:2].upper()} {i + 1}"

        price = random.randint(
            details["min_price"],
            details["max_price"]
        )

        # Generate a launch date
        start_date = datetime(2025, 1, 1)
        end_date = datetime(2026, 8, 1)

        days_between = (end_date - start_date).days

        random_days = random.randint(0, days_between)

        launch_date = start_date + timedelta(days=random_days)

        products.append({
            "model_id": model_id,
            "model_name": model_name,
            "price": price,
            "segment": segment,
            "launch_date": launch_date.date()
        })

        product_number += 1

# Create DataFrame

products_df = pd.DataFrame(products)

# Sort by launch date

products_df = products_df.sort_values(
    "launch_date"
).reset_index(drop=True)

# Save dataset

products_df.to_csv(
    "data/products.csv",
    index=False
)

# Check output

print("Products generated successfully!")
print()

print("Total products:", len(products_df))

print("\nProducts by segment:")
print(products_df["segment"].value_counts())

print("\nFirst 10 products:")
print(products_df.head(10))

Products generated successfully!

Total products: 60

Products by segment:
segment
Budget       20
Mid-range    20
Premium      12
Flagship      8
Name: count, dtype: int64

First 10 products:
  model_id  model_name   price    segment launch_date
0     P014  Mobi BU 14   10206     Budget  2025-01-13
1     P038  Mobi MI 18   20282  Mid-range  2025-01-20
2     P050  Mobi PR 10   47926    Premium  2025-01-26
3     P029   Mobi MI 9   27143  Mid-range  2025-02-04
4     P012  Mobi BU 12   10340     Budget  2025-02-11
5     P056   Mobi FL 4  146173   Flagship  2025-03-14
6     P059   Mobi FL 7  132723   Flagship  2025-03-16
7     P053   Mobi FL 1   78449   Flagship  2025-03-20
8     P024   Mobi MI 4   26314  Mid-range  2025-03-24
9     P004   Mobi BU 4    9027     Budget  2025-03-25


In [4]:
os.makedirs("data", exist_ok=True)

products_df.to_csv(
    "data/products.csv",
    index=False
)

In [5]:
print("Products saved successfully!")
print("Total products:", len(products_df))
print(products_df.head())

Products saved successfully!
Total products: 60
  model_id  model_name  price    segment launch_date
0     P014  Mobi BU 14  10206     Budget  2025-01-13
1     P038  Mobi MI 18  20282  Mid-range  2025-01-20
2     P050  Mobi PR 10  47926    Premium  2025-01-26
3     P029   Mobi MI 9  27143  Mid-range  2025-02-04
4     P012  Mobi BU 12  10340     Budget  2025-02-11


## 3. Generate Store Data

In [7]:
stores_df = pd.read_csv("data/stores.csv")

print("Total stores:", len(stores_df))
print(stores_df.head())

Total stores: 25
  store_id       city location_type store_size footfall customer_segment
0      S01  Bangalore       Premium      Large     High          Premium
1      S02  Bangalore    Commercial      Large     High          Premium
2      S03  Bangalore   Residential     Medium   Medium        Mid-range
3      S04  Bangalore    Commercial      Large     High          Premium
4      S05  Bangalore   Residential     Medium   Medium        Mid-range


In [8]:
def get_festival_factor(date):
    if date.month in [10, 11]:
        return 3.0
    return 1.0

In [9]:
start_date = "2025-09-01"
end_date = "2026-08-31"

dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

print("Total days:", len(dates))

Total days: 365


In [10]:
base_demand = {
    "Budget": 8,
    "Mid-range": 6,
    "Premium": 3,
    "Flagship": 2
}

store_factors = {
    "Budget": {
        "Budget": 1.5,
        "Mid-range": 1.1,
        "Premium": 0.7,
        "Flagship": 0.5
    },

    "Mid-range": {
        "Budget": 1.1,
        "Mid-range": 1.4,
        "Premium": 1.0,
        "Flagship": 0.7
    },

    "Premium": {
        "Budget": 0.7,
        "Mid-range": 1.0,
        "Premium": 1.4,
        "Flagship": 1.5
    }
}

In [11]:
import random

sales = []

for date in dates:

    for _, store in stores_df.iterrows():

        for _, product in products_df.iterrows():

            # Product and store segments
            product_segment = product["segment"]
            store_segment = store["customer_segment"]

            # Base demand
            base = base_demand[product_segment]

            # Store preference
            store_factor = store_factors[
                store_segment
            ][product_segment]

            # Festival effect
            festival_factor = get_festival_factor(date)

            # Random daily variation
            random_factor = random.uniform(0.8, 1.2)

            # Calculate expected demand
            expected_demand = (
                base
                * store_factor
                * festival_factor
                * random_factor
            )

            # Convert to whole units
            units_sold = max(
                0,
                int(round(expected_demand))
            )

            # Revenue
            revenue = (
                units_sold
                * product["price"]
            )

            # Store record
            sales.append({
                "date": date,
                "store_id": store["store_id"],
                "model_id": product["model_id"],
                "units_sold": units_sold,
                "selling_price": product["price"],
                "revenue": revenue
            })


sales_df = pd.DataFrame(sales)

print(sales_df.head())
print()
print("Total records:", len(sales_df))

        date store_id model_id  units_sold  selling_price  revenue
0 2025-09-01      S01     P014           6          10206    61236
1 2025-09-01      S01     P038           6          20282   121692
2 2025-09-01      S01     P050           4          47926   191704
3 2025-09-01      S01     P029           6          27143   162858
4 2025-09-01      S01     P012           7          10340    72380

Total records: 547500


In [12]:
# Apply successor cannibalization

# Create a lookup:
# model_id -> launch_date

launch_dates = dict(
    zip(
        products_df["model_id"],
        products_df["launch_date"]
    )
)


def get_cannibalization_factor(model_id, date):

    # Find successor
    product_row = products_df[
        products_df["model_id"] == model_id
    ]

    successor = product_row.iloc[0]["successor_model"]

    # No successor
    if pd.isna(successor):
        return 1.0

    # Successor launch date
    successor_launch = launch_dates[successor]

    # Convert to datetime
    successor_launch = pd.to_datetime(successor_launch)
    date = pd.to_datetime(date)

    # After successor launch
    if date >= successor_launch:
        return 0.5

    return 1.0

In [13]:
products_df["successor_model"] = None

successor_pairs = {
    "P001": "P002",
    "P003": "P004",
    "P005": "P006",

    "P021": "P022",
    "P023": "P024",
    "P025": "P026",

    "P041": "P042",
    "P043": "P044",

    "P053": "P054",
    "P055": "P056",
    "P057": "P058"
}

for old_model, new_model in successor_pairs.items():
    products_df.loc[
        products_df["model_id"] == old_model,
        "successor_model"
    ] = new_model

In [14]:
products_df.to_csv(
    "data/products.csv",
    index=False
)

In [15]:
# Fix invalid successor launch dates

products_df.loc[
    products_df["model_id"] == "P044",
    "launch_date"
] = "2026-05-12"

products_df.loc[
    products_df["model_id"] == "P002",
    "launch_date"
] = "2026-06-14"

products_df.loc[
    products_df["model_id"] == "P042",
    "launch_date"
] = "2026-08-17"

# Give P058 a more realistic gap from P057
products_df.loc[
    products_df["model_id"] == "P058",
    "launch_date"
] = "2026-08-13"

In [16]:
for _, row in products_df[
    products_df["successor_model"].notna()
].iterrows():

    old_model = row["model_id"]
    successor = row["successor_model"]

    old_date = row["launch_date"]

    new_date = products_df.loc[
        products_df["model_id"] == successor,
        "launch_date"
    ].iloc[0]

    print(
        old_model,
        "→",
        successor,
        "|",
        old_date,
        "→",
        new_date
    )

P053 → P054 | 2025-03-20 → 2026-02-27
P041 → P042 | 2025-05-12 → 2026-08-17
P055 → P056 | 2025-07-08 → 2025-03-14
P043 → P044 | 2025-08-29 → 2026-05-12
P003 → P004 | 2025-09-24 → 2025-03-25
P005 → P006 | 2025-10-21 → 2026-07-22
P025 → P026 | 2025-12-18 → 2026-04-21
P023 → P024 | 2025-12-25 → 2025-03-24
P057 → P058 | 2026-04-22 → 2026-08-13
P001 → P002 | 2026-05-07 → 2026-06-14
P021 → P022 | 2026-05-24 → 2026-04-16


In [17]:
products_df.to_csv(
    "data/products.csv",
    index=False
)

print("Products updated successfully!")
del sales_df

Products updated successfully!


In [18]:
successor_lookup = dict(
    zip(
        products_df["model_id"],
        products_df["successor_model"]
    )
)

launch_lookup = dict(
    zip(
        products_df["model_id"],
        pd.to_datetime(products_df["launch_date"])
    )
)

In [19]:
sales = []

for date in dates:

    for _, store in stores_df.iterrows():

        store_segment = store["customer_segment"]

        for _, product in products_df.iterrows():

            model_id = product["model_id"]
            product_segment = product["segment"]

            # -------------------------
            # 1. Base demand
            # -------------------------
            base = base_demand[product_segment]

            # -------------------------
            # 2. Store preference
            # -------------------------
            store_factor = store_factors[
                store_segment
            ][product_segment]

            # -------------------------
            # 3. Festival effect
            # -------------------------
            festival_factor = get_festival_factor(date)

            # -------------------------
            # 4. Successor effect
            # -------------------------
            cannibalization_factor = 1.0

            successor = successor_lookup.get(model_id)

            if pd.notna(successor):

                successor_launch = launch_lookup[successor]

                if date >= successor_launch:
                    cannibalization_factor = 0.5

            # -------------------------
            # 5. Random variation
            # -------------------------
            random_factor = random.uniform(
                0.8,
                1.2
            )

            # -------------------------
            # 6. Final demand
            # -------------------------
            expected_demand = (
                base
                * store_factor
                * festival_factor
                * cannibalization_factor
                * random_factor
            )

            units_sold = max(
                0,
                int(round(expected_demand))
            )

            # -------------------------
            # 7. Revenue
            # -------------------------
            revenue = (
                units_sold
                * product["price"]
            )

            sales.append({
                "date": date,
                "store_id": store["store_id"],
                "model_id": model_id,
                "units_sold": units_sold,
                "selling_price": product["price"],
                "revenue": revenue
            })


sales_df = pd.DataFrame(sales)

print(sales_df.head())
print("Total records:", len(sales_df))

        date store_id model_id  units_sold  selling_price  revenue
0 2025-09-01      S01     P014           7          10206    71442
1 2025-09-01      S01     P038           5          20282   101410
2 2025-09-01      S01     P050           5          47926   239630
3 2025-09-01      S01     P029           5          27143   135715
4 2025-09-01      S01     P012           5          10340    51700
Total records: 547500


In [20]:
sales_df["month"] = sales_df["date"].dt.month

monthly_sales = (
    sales_df
    .groupby("month")["units_sold"]
    .sum()
)

print(monthly_sales)

month
1     289989
2     261977
3     289502
4     277948
5     283005
6     271878
7     277858
8     274379
9     280657
10    871235
11    842523
12    290033
Name: units_sold, dtype: int64


In [21]:
sales_df["month"] = sales_df["date"].dt.month

monthly_sales = (
    sales_df
    .groupby("month")["units_sold"]
    .sum()
)

print(monthly_sales)

month
1     289989
2     261977
3     289502
4     277948
5     283005
6     271878
7     277858
8     274379
9     280657
10    871235
11    842523
12    290033
Name: units_sold, dtype: int64


In [22]:
sales_df.drop(columns=["month"], inplace=True)

print(sales_df.columns)

Index(['date', 'store_id', 'model_id', 'units_sold', 'selling_price',
       'revenue'],
      dtype='object')


In [23]:
old_model = "P001"
new_model = "P002"

new_launch_date = launch_lookup[new_model]

before = sales_df[
    (sales_df["model_id"] == old_model) &
    (sales_df["date"] < new_launch_date)
]["units_sold"].mean()

after = sales_df[
    (sales_df["model_id"] == old_model) &
    (sales_df["date"] >= new_launch_date)
]["units_sold"].mean()

print("Successor launch date:", new_launch_date)
print("P001 average sales BEFORE:", before)
print("P001 average sales AFTER:", after)

Successor launch date: 2026-06-14 00:00:00
P001 average sales BEFORE: 13.832027972027973
P001 average sales AFTER: 4.844050632911393


In [24]:
sales_df.to_csv(
    "data/sales_history.csv",
    index=False
)

print("sales_history.csv saved successfully!")

sales_history.csv saved successfully!


In [25]:
print(
    "File exists:",
    os.path.exists("data/sales_history.csv")
)

print(
    "File size:",
    os.path.getsize("data/sales_history.csv") / (1024 * 1024),
    "MB"
)

File exists: True
File size: 18.814462661743164 MB


In [26]:
import pandas as pd

sales_df = pd.read_csv(
    "data/sales_history.csv"
)

sales_df["date"] = pd.to_datetime(
    sales_df["date"]
)

print(sales_df.head())
print(sales_df.shape)


        date store_id model_id  units_sold  selling_price  revenue
0 2025-09-01      S01     P014           7          10206    71442
1 2025-09-01      S01     P038           5          20282   101410
2 2025-09-01      S01     P050           5          47926   239630
3 2025-09-01      S01     P029           5          27143   135715
4 2025-09-01      S01     P012           5          10340    51700
(547500, 6)


In [27]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 547500 entries, 0 to 547499
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   date           547500 non-null  datetime64[ns]
 1   store_id       547500 non-null  object        
 2   model_id       547500 non-null  object        
 3   units_sold     547500 non-null  int64         
 4   selling_price  547500 non-null  int64         
 5   revenue        547500 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 25.1+ MB


In [28]:
sales_df[
    ["units_sold", "selling_price", "revenue"]
].describe()

product_sales = (
    sales_df
    .groupby("model_id")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(product_sales.head(10))

model_id
P008    118370
P014    118187
P011    118167
P012    118089
P020    118042
P010    117983
P015    117981
P016    117973
P007    117928
P017    117921
Name: units_sold, dtype: int64


In [29]:
store_sales = (
    sales_df
    .groupby("store_id")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(store_sales.head(10))

store_id
S14    190713
S11    190684
S16    190613
S17    190594
S20    190576
S22    190562
S12    190340
S18    190324
S25    190315
S19    190301
Name: units_sold, dtype: int64


In [30]:
sales_with_products = sales_df.merge(
    products_df[
        ["model_id", "segment", "price"]
    ],
    on="model_id",
    how="left"
)


segment_sales = (
    sales_with_products
    .groupby("segment")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(segment_sales)

segment
Budget       2285959
Mid-range    1691831
Premium       401623
Flagship      131571
Name: units_sold, dtype: int64


In [31]:
print(products_df.columns.tolist())

['model_id', 'model_name', 'price', 'segment', 'launch_date', 'successor_model']


In [32]:
sales_with_products = sales_df.merge(
    products_df[
        ["model_id", "segment", "price"]
    ],
    on="model_id",
    how="left"
)

print(sales_with_products.head())
print(sales_with_products.shape)

        date store_id model_id  units_sold  selling_price  revenue    segment  \
0 2025-09-01      S01     P014           7          10206    71442     Budget   
1 2025-09-01      S01     P038           5          20282   101410  Mid-range   
2 2025-09-01      S01     P050           5          47926   239630    Premium   
3 2025-09-01      S01     P029           5          27143   135715  Mid-range   
4 2025-09-01      S01     P012           5          10340    51700     Budget   

   price  
0  10206  
1  20282  
2  47926  
3  27143  
4  10340  
(547500, 8)


In [33]:
segment_sales = (
    sales_with_products
    .groupby("segment")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(segment_sales)


product_sales = (
    sales_df
    .groupby("model_id")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(product_sales.head(10))

segment
Budget       2285959
Mid-range    1691831
Premium       401623
Flagship      131571
Name: units_sold, dtype: int64
model_id
P008    118370
P014    118187
P011    118167
P012    118089
P020    118042
P010    117983
P015    117981
P016    117973
P007    117928
P017    117921
Name: units_sold, dtype: int64


In [34]:
store_sales = (
    sales_df
    .groupby("store_id")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

print(store_sales.head(10))

monthly_sales = (
    sales_df.assign(
        month=sales_df["date"].dt.to_period("M")
    )
    .groupby("month")["units_sold"]
    .sum()
)

print(monthly_sales)

store_id
S14    190713
S11    190684
S16    190613
S17    190594
S20    190576
S22    190562
S12    190340
S18    190324
S25    190315
S19    190301
Name: units_sold, dtype: int64
month
2025-09    280657
2025-10    871235
2025-11    842523
2025-12    290033
2026-01    289989
2026-02    261977
2026-03    289502
2026-04    277948
2026-05    283005
2026-06    271878
2026-07    277858
2026-08    274379
Freq: M, Name: units_sold, dtype: int64


In [35]:
store_segment_sales = (
    sales_with_products
    .groupby(["store_id", "segment"])["units_sold"]
    .sum()
    .reset_index()
)

print(store_segment_sales.head(20))

segment_by_store = store_segment_sales.pivot(
    index="store_id",
    columns="segment",
    values="units_sold"
)

print(segment_by_store)

   store_id    segment  units_sold
0       S01     Budget       52912
1       S01   Flagship       10648
2       S01  Mid-range       56083
3       S01    Premium       24501
4       S02     Budget       52965
5       S02   Flagship       10712
6       S02  Mid-range       56140
7       S02    Premium       24404
8       S03     Budget       82783
9       S03   Flagship        4980
10      S03  Mid-range       78493
11      S03    Premium       17384
12      S04     Budget       53021
13      S04   Flagship       10693
14      S04  Mid-range       56263
15      S04    Premium       24514
16      S05     Budget       82968
17      S05   Flagship        5015
18      S05  Mid-range       78556
19      S05    Premium       17340
segment   Budget  Flagship  Mid-range  Premium
store_id                                      
S01        52912     10648      56083    24501
S02        52965     10712      56140    24404
S03        82783      4980      78493    17384
S04        53021     10693    

In [36]:
forecast_df = sales_df.copy()

forecast_df["day_of_week"] = (
    forecast_df["date"].dt.dayofweek
)

forecast_df["month"] = (
    forecast_df["date"].dt.month
)

forecast_df["week_of_year"] = (
    forecast_df["date"].dt.isocalendar().week
)

forecast_df["day_of_month"] = (
    forecast_df["date"].dt.day
)

print(forecast_df.head())



        date store_id model_id  units_sold  selling_price  revenue  \
0 2025-09-01      S01     P014           7          10206    71442   
1 2025-09-01      S01     P038           5          20282   101410   
2 2025-09-01      S01     P050           5          47926   239630   
3 2025-09-01      S01     P029           5          27143   135715   
4 2025-09-01      S01     P012           5          10340    51700   

   day_of_week  month  week_of_year  day_of_month  
0            0      9            36             1  
1            0      9            36             1  
2            0      9            36             1  
3            0      9            36             1  
4            0      9            36             1  


In [37]:
forecast_df = forecast_df.merge(
    products_df[
        [
            "model_id",
            "segment",
            "price",
            "launch_date",
            "successor_model"
        ]
    ],
    on="model_id",
    how="left"
)

print(forecast_df.head())
print(forecast_df.shape)

        date store_id model_id  units_sold  selling_price  revenue  \
0 2025-09-01      S01     P014           7          10206    71442   
1 2025-09-01      S01     P038           5          20282   101410   
2 2025-09-01      S01     P050           5          47926   239630   
3 2025-09-01      S01     P029           5          27143   135715   
4 2025-09-01      S01     P012           5          10340    51700   

   day_of_week  month  week_of_year  day_of_month    segment  price  \
0            0      9            36             1     Budget  10206   
1            0      9            36             1  Mid-range  20282   
2            0      9            36             1    Premium  47926   
3            0      9            36             1  Mid-range  27143   
4            0      9            36             1     Budget  10340   

  launch_date successor_model  
0  2025-01-13            None  
1  2025-01-20            None  
2  2025-01-26            None  
3  2025-02-04           

In [38]:
forecast_df = forecast_df.sort_values(
    ["store_id", "model_id", "date"]
).reset_index(drop=True)

print(
    forecast_df[
        ["date", "store_id", "model_id", "units_sold"]
    ].head(10)
)

        date store_id model_id  units_sold
0 2025-09-01      S01     P001           6
1 2025-09-02      S01     P001           6
2 2025-09-03      S01     P001           6
3 2025-09-04      S01     P001           6
4 2025-09-05      S01     P001           5
5 2025-09-06      S01     P001           7
6 2025-09-07      S01     P001           5
7 2025-09-08      S01     P001           6
8 2025-09-09      S01     P001           6
9 2025-09-10      S01     P001           7


In [39]:
forecast_df["rolling_7d"] = (
    forecast_df
    .groupby(["store_id", "model_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

print(forecast_df.shape)
print(forecast_df["rolling_7d"].isnull().sum())

(547500, 15)
10500


In [40]:
print(
    forecast_df[
        [
            "date",
            "store_id",
            "model_id",
            "units_sold",
            "rolling_7d"
        ]
    ].head(15)
)

         date store_id model_id  units_sold  rolling_7d
0  2025-09-01      S01     P001           6         NaN
1  2025-09-02      S01     P001           6         NaN
2  2025-09-03      S01     P001           6         NaN
3  2025-09-04      S01     P001           6         NaN
4  2025-09-05      S01     P001           5         NaN
5  2025-09-06      S01     P001           7         NaN
6  2025-09-07      S01     P001           5         NaN
7  2025-09-08      S01     P001           6    5.857143
8  2025-09-09      S01     P001           6    5.857143
9  2025-09-10      S01     P001           7    5.857143
10 2025-09-11      S01     P001           6    6.000000
11 2025-09-12      S01     P001           6    6.000000
12 2025-09-13      S01     P001           6    6.142857
13 2025-09-14      S01     P001           5    6.000000
14 2025-09-15      S01     P001           5    6.000000


In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

baseline_df = forecast_df.dropna(
    subset=["rolling_7d"]
).copy()

baseline_df["baseline_prediction"] = (
    baseline_df["rolling_7d"]
)

mae = mean_absolute_error(
    baseline_df["units_sold"],
    baseline_df["baseline_prediction"]
)

rmse = np.sqrt(
    mean_squared_error(
        baseline_df["units_sold"],
        baseline_df["baseline_prediction"]
    )
)

print("Baseline MAE:", mae)
print("Baseline RMSE:", rmse)

Baseline MAE: 1.117749401436552
Baseline RMSE: 2.2737266287473785


In [42]:
# Sort by date
forecast_df = forecast_df.sort_values("date").reset_index(drop=True)

# Training: up to June 2026
train_df = forecast_df[
    forecast_df["date"] < "2026-07-01"
].copy()

# Testing: July + August 2026
test_df = forecast_df[
    forecast_df["date"] >= "2026-07-01"
].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (454500, 15)
Test: (93000, 15)


In [43]:
features = [
    "store_id",
    "model_id",
    "segment",
    "price",
    "day_of_week",
    "month",
    "week_of_year",
    "day_of_month",
    "rolling_7d"
]

target = "units_sold"

X = forecast_df[features].copy()
y = forecast_df[target].copy()

X = pd.get_dummies(
    X,
    columns=["store_id", "model_id", "segment"],
    dtype=int
)

print(X.shape)



(547500, 95)


In [44]:
from sklearn.ensemble import RandomForestRegressor

# Prepare train/test features
X_train = X.loc[train_df.index]
X_test = X.loc[test_df.index]

y_train = train_df["units_sold"]
y_test = test_df["units_sold"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (454500, 95)
X_test: (93000, 95)


In [45]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(
    X_train,
    y_train
)

print("Model training completed!")

Model training completed!


In [46]:
rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)

Random Forest MAE: 0.6541337404261909
Random Forest RMSE: 0.9903047325202159


In [47]:
comparison = pd.DataFrame({
    "Model": [
        "7-Day Baseline",
        "Random Forest"
    ],
    "MAE": [
        mae,
        rf_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse
    ]
})

print(comparison)

            Model       MAE      RMSE
0  7-Day Baseline  1.117749  2.273727
1   Random Forest  0.654134  0.990305


In [48]:
import joblib

joblib.dump(
    rf_model,
    "random_forest_forecaster.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [49]:
allocation_df = test_df[
    [
        "date",
        "store_id",
        "model_id",
        "segment",
        "price",
        "units_sold"
    ]
].copy()

allocation_df["forecast_demand"] = rf_predictions

allocation_df["forecast_demand"] = (
    allocation_df["forecast_demand"]
    .clip(lower=0)
    .round()
    .astype(int)
)

print(allocation_df.head())
print(allocation_df.shape)

             date store_id model_id    segment  price  units_sold  \
454500 2026-07-01      S07     P037  Mid-range  26798           8   
454501 2026-07-01      S11     P050    Premium  47926           2   
454502 2026-07-01      S06     P040  Mid-range  20298           5   
454503 2026-07-01      S01     P024  Mid-range  26314           7   
454504 2026-07-01      S15     P025  Mid-range  21573           5   

        forecast_demand  
454500                9  
454501                2  
454502                6  
454503                6  
454504                4  
(93000, 7)


In [50]:
store_model_forecast = (
    allocation_df
    .groupby(
        ["store_id", "model_id", "segment"],
        as_index=False
    )["forecast_demand"]
    .sum()
)

print(store_model_forecast.head(20))

   store_id model_id segment  forecast_demand
0       S01     P001  Budget              187
1       S01     P002  Budget              372
2       S01     P003  Budget              187
3       S01     P004  Budget              372
4       S01     P005  Budget              264
5       S01     P006  Budget              372
6       S01     P007  Budget              372
7       S01     P008  Budget              372
8       S01     P009  Budget              372
9       S01     P010  Budget              370
10      S01     P011  Budget              372
11      S01     P012  Budget              372
12      S01     P013  Budget              372
13      S01     P014  Budget              372
14      S01     P015  Budget              372
15      S01     P016  Budget              372
16      S01     P017  Budget              372
17      S01     P018  Budget              372
18      S01     P019  Budget              372
19      S01     P020  Budget              372


In [51]:
store_model_forecast["safety_stock"] = (
    store_model_forecast["forecast_demand"] * 0.20
).round().astype(int)

store_model_forecast["recommended_inventory"] = (
    store_model_forecast["forecast_demand"]
    + store_model_forecast["safety_stock"]
)

print(store_model_forecast.head(20))

   store_id model_id segment  forecast_demand  safety_stock  \
0       S01     P001  Budget              187            37   
1       S01     P002  Budget              372            74   
2       S01     P003  Budget              187            37   
3       S01     P004  Budget              372            74   
4       S01     P005  Budget              264            53   
5       S01     P006  Budget              372            74   
6       S01     P007  Budget              372            74   
7       S01     P008  Budget              372            74   
8       S01     P009  Budget              372            74   
9       S01     P010  Budget              370            74   
10      S01     P011  Budget              372            74   
11      S01     P012  Budget              372            74   
12      S01     P013  Budget              372            74   
13      S01     P014  Budget              372            74   
14      S01     P015  Budget              372          

In [52]:
store_model_forecast.to_csv(
    "data/store_model_forecast.csv",
    index=False
)

print("store_model_forecast.csv saved successfully!")

store_model_forecast.csv saved successfully!


In [53]:
# Total recommended inventory for each model
model_inventory = (
    store_model_forecast
    .groupby("model_id")["recommended_inventory"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "recommended_inventory": "total_required_inventory"
        }
    )
)

print(model_inventory.head(20))

   model_id  total_required_inventory
0      P001                      9023
1      P002                     18438
2      P003                      9058
3      P004                     18436
4      P005                     13097
5      P006                     18420
6      P007                     18434
7      P008                     18431
8      P009                     18435
9      P010                     18436
10     P011                     18438
11     P012                     18440
12     P013                     18433
13     P014                     18439
14     P015                     18432
15     P016                     18440
16     P017                     18439
17     P018                     18437
18     P019                     18418
19     P020                     18441


In [54]:
# Store-level inventory requirement
store_inventory = (
    store_model_forecast
    .groupby("store_id")["recommended_inventory"]
    .sum()
    .sort_values(ascending=False)
)

print(store_inventory.head(10))

store_id
S16    28698
S17    28689
S14    28662
S18    28652
S12    28645
S25    28637
S11    28624
S19    28621
S21    28609
S22    28594
Name: recommended_inventory, dtype: int32


In [55]:
model_inventory.to_csv(
    "data/model_inventory_requirement.csv",
    index=False
)

store_inventory.to_csv(
    "data/store_inventory_requirement.csv"
)

print("Inventory requirement files saved!")

Inventory requirement files saved!


In [56]:
import numpy as np

np.random.seed(42)

store_model_forecast["current_inventory"] = (
    store_model_forecast["recommended_inventory"]
    * np.random.uniform(0.7, 1.3, len(store_model_forecast))
).round().astype(int)

print(
    store_model_forecast[
        [
            "store_id",
            "model_id",
            "recommended_inventory",
            "current_inventory"
        ]
    ].head(20)
)

   store_id model_id  recommended_inventory  current_inventory
0       S01     P001                    224                207
1       S01     P002                    446                567
2       S01     P003                    224                255
3       S01     P004                    446                472
4       S01     P005                    317                252
5       S01     P006                    446                354
6       S01     P007                    446                328
7       S01     P008                    446                544
8       S01     P009                    446                473
9       S01     P010                    444                499
10      S01     P011                    446                318
11      S01     P012                    446                572
12      S01     P013                    446                535
13      S01     P014                    446                369
14      S01     P015                    446            

In [57]:
store_model_forecast["inventory_gap"] = (
    store_model_forecast["current_inventory"]
    - store_model_forecast["recommended_inventory"]
)

store_model_forecast["excess_inventory"] = (
    store_model_forecast["inventory_gap"]
    .clip(lower=0)
)

store_model_forecast["shortage_inventory"] = (
    -store_model_forecast["inventory_gap"]
    .clip(upper=0)
)

print(
    store_model_forecast[
        [
            "store_id",
            "model_id",
            "recommended_inventory",
            "current_inventory",
            "excess_inventory",
            "shortage_inventory"
        ]
    ].head(20)
)

   store_id model_id  recommended_inventory  current_inventory  \
0       S01     P001                    224                207   
1       S01     P002                    446                567   
2       S01     P003                    224                255   
3       S01     P004                    446                472   
4       S01     P005                    317                252   
5       S01     P006                    446                354   
6       S01     P007                    446                328   
7       S01     P008                    446                544   
8       S01     P009                    446                473   
9       S01     P010                    444                499   
10      S01     P011                    446                318   
11      S01     P012                    446                572   
12      S01     P013                    446                535   
13      S01     P014                    446                369   
14      S0

In [58]:
print(
    "Total required inventory:",
    store_model_forecast["recommended_inventory"].sum()
)

print(
    "Total current inventory:",
    store_model_forecast["current_inventory"].sum()
)

print(
    "Total excess inventory:",
    store_model_forecast["excess_inventory"].sum()
)

print(
    "Total shortage:",
    store_model_forecast["shortage_inventory"].sum()
)

Total required inventory: 669784
Total current inventory: 670637
Total excess inventory: 51768
Total shortage: 50915


In [59]:
excess_df = store_model_forecast[
    store_model_forecast["excess_inventory"] > 0
].copy()

shortage_df = store_model_forecast[
    store_model_forecast["shortage_inventory"] > 0
].copy()

print("Excess records:", len(excess_df))
print("Shortage records:", len(shortage_df))

transfers = []

for model_id in store_model_forecast["model_id"].unique():

    model_excess = excess_df[
        excess_df["model_id"] == model_id
    ].copy()

    model_shortage = shortage_df[
        shortage_df["model_id"] == model_id
    ].copy()

    for shortage_idx, shortage_row in model_shortage.iterrows():

        remaining_shortage = shortage_row["shortage_inventory"]

        for excess_idx, excess_row in model_excess.iterrows():

            if remaining_shortage <= 0:
                break

            available_excess = excess_row["excess_inventory"]

            transfer_qty = min(
                remaining_shortage,
                available_excess
            )

            if transfer_qty > 0:

                transfers.append({
                    "model_id": model_id,
                    "from_store": excess_row["store_id"],
                    "to_store": shortage_row["store_id"],
                    "transfer_quantity": int(transfer_qty)
                })

                remaining_shortage -= transfer_qty

                model_excess.loc[
                    excess_idx,
                    "excess_inventory"
                ] -= transfer_qty

transfers_df = pd.DataFrame(transfers)

print(transfers_df.head(20))
print("Total transfer records:", len(transfers_df))

Excess records: 757
Shortage records: 733
   model_id from_store to_store  transfer_quantity
0      P001        S03      S01                 17
1      P001        S03      S02                 15
2      P001        S03      S04                 21
3      P001        S03      S06                  4
4      P001        S05      S06                 56
5      P001        S05      S07                 21
6      P001        S05      S11                  5
7      P001        S08      S11                 54
8      P001        S09      S11                 24
9      P001        S10      S11                  9
10     P001        S10      S12                 39
11     P001        S10      S15                 37
12     P001        S13      S15                 54
13     P001        S13      S16                 27
14     P001        S14      S16                 51
15     P001        S14      S17                  4
16     P001        S18      S17                 53
17     P001        S19      S17         

In [60]:
transfers_df.to_csv(
    "data/inventory_transfers.csv",
    index=False
)

print("inventory_transfers.csv saved successfully!")

inventory_transfers.csv saved successfully!


In [61]:
eol_df = products_df.copy()

eol_df["launch_date"] = pd.to_datetime(
    eol_df["launch_date"]
)

eol_df["is_eol"] = eol_df.apply(
    lambda row:
        pd.notna(row["successor_model"]) and
        row["successor_model"] in products_df["model_id"].values,
    axis=1
)

print(
    eol_df[
        [
            "model_id",
            "model_name",
            "launch_date",
            "successor_model",
            "is_eol"
        ]
    ].head(20)
)


eol_models = eol_df[
    eol_df["is_eol"]
]["model_id"].tolist()

eol_inventory = store_model_forecast[
    store_model_forecast["model_id"].isin(eol_models)
].copy()

print("EOL models:", len(eol_models))
print("EOL inventory:", len(eol_inventory))

   model_id  model_name launch_date successor_model  is_eol
0      P014  Mobi BU 14  2025-01-13            None   False
1      P038  Mobi MI 18  2025-01-20            None   False
2      P050  Mobi PR 10  2025-01-26            None   False
3      P029   Mobi MI 9  2025-02-04            None   False
4      P012  Mobi BU 12  2025-02-11            None   False
5      P056   Mobi FL 4  2025-03-14            None   False
6      P059   Mobi FL 7  2025-03-16            None   False
7      P053   Mobi FL 1  2025-03-20            P054    True
8      P024   Mobi MI 4  2025-03-24            None   False
9      P004   Mobi BU 4  2025-03-25            None   False
10     P040  Mobi MI 20  2025-04-05            None   False
11     P047   Mobi PR 7  2025-04-06            None   False
12     P039  Mobi MI 19  2025-04-17            None   False
13     P041   Mobi PR 1  2025-05-12            P042    True
14     P045   Mobi PR 5  2025-05-20            None   False
15     P048   Mobi PR 8  2025-06-03     

In [62]:
eol_inventory["recommendation"] = np.where(
    eol_inventory["excess_inventory"] > 0,
    "MARKDOWN",
    "NORMAL"
)

print(
    eol_inventory[
        [
            "store_id",
            "model_id",
            "excess_inventory",
            "recommendation"
        ]
    ].head(20)
)

    store_id model_id  excess_inventory recommendation
0        S01     P001                 0         NORMAL
2        S01     P003                31       MARKDOWN
4        S01     P005                 0         NORMAL
20       S01     P021                15       MARKDOWN
22       S01     P023                 0         NORMAL
24       S01     P025                 0         NORMAL
40       S01     P041                 0         NORMAL
42       S01     P043                 0         NORMAL
52       S01     P053                22       MARKDOWN
54       S01     P055                 6       MARKDOWN
56       S01     P057                 0         NORMAL
60       S02     P001                 0         NORMAL
62       S02     P003                44       MARKDOWN
64       S02     P005                 0         NORMAL
80       S02     P021                49       MARKDOWN
82       S02     P023                 0         NORMAL
84       S02     P025                 0         NORMAL
100      S

In [63]:
eol_inventory.to_csv(
    "data/eol_markdown_recommendations.csv",
    index=False
)

print("EOL recommendations saved!")

EOL recommendations saved!


In [64]:
# Final KPI Summary

total_required = store_model_forecast[
    "recommended_inventory"
].sum()

total_current = store_model_forecast[
    "current_inventory"
].sum()

total_excess = store_model_forecast[
    "excess_inventory"
].sum()

total_shortage = store_model_forecast[
    "shortage_inventory"
].sum()

total_transfers = transfers_df[
    "transfer_quantity"
].sum()

total_markdown = eol_inventory[
    eol_inventory["recommendation"] == "MARKDOWN"
]["excess_inventory"].sum()

print("========== MobiMart KPI Summary ==========")

print(
    "Total Required Inventory:",
    total_required
)

print(
    "Total Current Inventory:",
    total_current
)

print(
    "Total Excess Inventory:",
    total_excess
)

print(
    "Total Shortage Inventory:",
    total_shortage
)

print(
    "Total Transfer Quantity:",
    total_transfers
)

print(
    "Total EOL Markdown Inventory:",
    total_markdown
)

print(
    "Number of Transfer Recommendations:",
    len(transfers_df)
)

print(
    "Number of EOL Markdown Recommendations:",
    len(
        eol_inventory[
            eol_inventory["recommendation"] == "MARKDOWN"
        ]
    )
)

========== MobiMart KPI Summary ==========
Total Required Inventory: 669784
Total Current Inventory: 670637
Total Excess Inventory: 51768
Total Shortage Inventory: 50915
Total Transfer Quantity: 41928
Total EOL Markdown Inventory: 4768
Number of Transfer Recommendations: 1188
Number of EOL Markdown Recommendations: 133


In [65]:
store_model_forecast.to_csv(
    "data/final_inventory_allocation.csv",
    index=False
)

transfers_df.to_csv(
    "data/inventory_transfers.csv",
    index=False
)

eol_inventory.to_csv(
    "data/eol_markdown_recommendations.csv",
    index=False
)

comparison.to_csv(
    "data/model_comparison.csv",
    index=False
)

print("All final files saved successfully!")

All final files saved successfully!


In [66]:
import os

files_to_check = [
    "data/products.csv",
    "data/stores.csv",
    "data/sales_history.csv",
    "data/store_model_forecast.csv",
    "data/model_inventory_requirement.csv",
    "data/store_inventory_requirement.csv",
    "data/inventory_transfers.csv",
    "data/eol_markdown_recommendations.csv",
    "data/final_inventory_allocation.csv",
    "data/model_comparison.csv",
    "random_forest_forecaster.pkl"
]

for file in files_to_check:
    print(
        file,
        "→",
        "OK" if os.path.exists(file) else "MISSING"
    )

data/products.csv → OK
data/stores.csv → OK
data/sales_history.csv → OK
data/store_model_forecast.csv → OK
data/model_inventory_requirement.csv → OK
data/store_inventory_requirement.csv → OK
data/inventory_transfers.csv → OK
data/eol_markdown_recommendations.csv → OK
data/final_inventory_allocation.csv → OK
data/model_comparison.csv → OK
random_forest_forecaster.pkl → OK


In [67]:
print("\nModel Comparison:")
print(comparison)


Model Comparison:
            Model       MAE      RMSE
0  7-Day Baseline  1.117749  2.273727
1   Random Forest  0.654134  0.990305


In [68]:
print("========== FINAL VALIDATION ==========")

print("Sales rows:", len(sales_df))
print("Sales units:", sales_df["units_sold"].sum())
print("Sales revenue:", sales_df["revenue"].sum())

print("Forecast rows:", len(forecast_df))

print("Baseline MAE:", round(mae, 4))
print("Baseline RMSE:", round(rmse, 4))

print("Random Forest MAE:", round(rf_mae, 4))
print("Random Forest RMSE:", round(rf_rmse, 4))

print("Transfer recommendations:", len(transfers_df))
print("Transfer quantity:", transfers_df["transfer_quantity"].sum())

print("EOL markdown recommendations:",
      len(eol_inventory[
          eol_inventory["recommendation"] == "MARKDOWN"
      ]))

print("======================================")
print("PROJECT VALIDATION COMPLETE")

========== FINAL VALIDATION ==========
Sales rows: 547500
Sales units: 4510984
Sales revenue: 102379214913
Forecast rows: 547500
Baseline MAE: 1.1177
Baseline RMSE: 2.2737
Random Forest MAE: 0.6541
Random Forest RMSE: 0.9903
Transfer recommendations: 1188
Transfer quantity: 41928
EOL markdown recommendations: 133
PROJECT VALIDATION COMPLETE


In [134]:
import os

print("Current folder:")
print(os.getcwd())

print("\nProject files:")
for item in os.listdir():
    print(item)

Current folder:
C:\Users\pc\scrape_data

Project files:
.ipynb_checkpoints
ANN.ipynb
app.py
best_model.pt
credit.ipynb
data
data_gen.ipynb
datefruit_ANN.ipynb
DateFruit_Dataset.csv
DT.ipynb
gradient_reg.ipynb
health_model.pkl
hierarchical_clustering.ipynb
holistic_health_lifestyle_with_steps.csv
IMDB Dataset.csv
insurance.csv
iris.ipynb
kmeans.ipynb
loan_approval_data.csv
matplotlib.ipynb
Online Retail.xlsx
powerplant_data.csv
random_forest_forecaster.pkl
README.md
requirements.txt
requirements.txt.csv
RNN.ipynb
Smartcart.ipynb
smartcart_customers.csv
SVC.ipynb
svr_implement.ipynb
thyroid_dataset.csv
Untitled Folder
Untitled.ipynb
Untitled1.ipynb
Untitled2.ipynb
Untitled4.ipynb
voting.ipynb


In [138]:
import os

mobi_folder = "MobiMart"

os.makedirs(mobi_folder, exist_ok=True)

print("MobiMart folder created!")
print(os.path.abspath(mobi_folder))

MobiMart folder created!
C:\Users\pc\scrape_data\MobiMart
